# Histopathologic Cancer Detection using Deep Learning

## Project Overview

This project focuses on building deep learning models to detect metastatic cancer in histopathologic images. The task is formulated as a binary image classification problem, where each image patch is classified as either containing cancerous tissue or not.

We use a subset of the Histopathologic Cancer Detection dataset from Kaggle and apply multiple deep learning strategies, including custom CNNs and transfer learning models, to evaluate performance under different training conditions.

## Background: Breast Cancer and H&E Staining

Breast cancer is one of the most common types of cancer worldwide and early detection plays a critical role in improving patient outcomes. Histopathologic analysis is a standard method used by medical professionals to examine tissue samples under a microscope to identify cancerous cells.

The dataset used in this project consists of small image patches extracted from lymph node sections. These images are stained using Hematoxylin and Eosin (H&E) staining, which is a widely used technique in pathology.

- **Hematoxylin** stains cell nuclei in blue or purple, highlighting genetic material.
- **Eosin** stains cytoplasm and surrounding tissue in pink.

This contrast helps in identifying abnormal cell structures and detecting the presence of metastatic cancer. However, manual analysis can be time-consuming and subjective, which motivates the use of deep learning models to automate and improve diagnostic accuracy.

## Project Requirements and Strategy

### Requirements

The project involves the following key requirements:

1. Test at least two models (one must be built from scratch).
2. Use at least two optimizers and two learning rate schedulers.
3. Apply at least two image augmentation techniques.
4. Generate ROC curves for different models and training configurations.
5. Perform hyperparameter tuning to improve model performance.

### Strategy

To satisfy these requirements, the following approach is adopted:

- Use a subset of 5000 images for efficient experimentation.
- Build a **Custom CNN** model from scratch as the baseline.
- Implement a **second model using transfer learning (ResNet18)**.
- Train each model under multiple configurations:
  - Optimizer: Adam and SGD
  - Scheduler: StepLR and CosineAnnealingLR
  - Augmentation: Rotation/Flip and Color Jitter
- Evaluate models using:
  - Loss and Accuracy curves
  - ROC-AUC scores
  - ROC curve comparisons

This structured experimentation allows us to systematically analyze how different training strategies affect model performance.

## Data Loading and Setup

The dataset used in this project is the **Histopathologic Cancer Detection** dataset from Kaggle. It contains small histopathology image patches labeled for the presence of metastatic cancer.

We use Google Colab along with the Kaggle API to download and extract the dataset. After extraction, the dataset consists of:

- A `train` folder containing labeled images
- A `test` folder containing unlabeled images
- A `train_labels.csv` file mapping image IDs to binary labels

In the following steps, we configure Kaggle access, download the dataset, and extract it for further processing.

In [ ]:
# Import the basic libraries required for setup, file handling, data processing, visualization, and model training

import os
import random
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    classification_report
)

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

In [ ]:
# Upload your kaggle.json file in Colab so the notebook can access the Kaggle API

from google.colab import files

uploaded = files.upload()

In [ ]:
# Save kaggle.json to the correct location and set permissions for Kaggle API access

import os

os.makedirs("/root/.kaggle", exist_ok=True)

with open("/root/.kaggle/kaggle.json", "wb") as f:
    f.write(uploaded["kaggle.json"])

os.chmod("/root/.kaggle/kaggle.json", 0o600)

print("Kaggle setup complete ")

In [ ]:
# Download the Histopathologic Cancer Detection dataset from Kaggle

!kaggle competitions download -c histopathologic-cancer-detection

In [ ]:
# Extract the downloaded dataset and define paths for train and test directories

import zipfile
import os

DATA_DIR = "data"

with zipfile.ZipFile("histopathologic-cancer-detection.zip", "r") as zip_ref:
    zip_ref.extractall(DATA_DIR)

print("Extraction complete ")

TRAIN_DIR = os.path.join(DATA_DIR, "train")
TEST_DIR  = os.path.join(DATA_DIR, "test")

print("Train directory:", TRAIN_DIR)
print("Test directory:", TEST_DIR)
print("Train images:", len(os.listdir(TRAIN_DIR)))
print("Test images:", len(os.listdir(TEST_DIR)))

## Data Exploration

Before building models, it is important to visually inspect the dataset to understand the structure and characteristics of the histopathologic images.

In this step, we randomly sample a few images from the training dataset to observe:
- Image quality and resolution
- Color distribution from H&E staining
- Visual differences between samples

This helps ensure that preprocessing and augmentation strategies are appropriate.

In [ ]:
# Visualize a few random images from the training dataset

import random
import matplotlib.pyplot as plt
from PIL import Image

sample_imgs = random.sample(os.listdir(TRAIN_DIR), 6)

plt.figure(figsize=(12, 6))

for i, img_name in enumerate(sample_imgs):
    img_path = os.path.join(TRAIN_DIR, img_name)
    img = Image.open(img_path)

    plt.subplot(2, 3, i+1)
    plt.imshow(img)
    plt.title(img_name[:10])
    plt.axis("off")

plt.tight_layout()
plt.show()

## Label Mapping and Subset Selection

The training images are associated with binary labels stored in a CSV file. Each image ID is mapped to a label indicating whether metastatic cancer is present or absent.

Since the full dataset is large, a stratified subset of 5000 images is selected for efficient experimentation while preserving the original class distribution. This allows faster training and easier comparison of different models and training configurations.

In [ ]:
# Load the label file and inspect the class distribution of the dataset

LABELS_CSV = os.path.join(DATA_DIR, "train_labels.csv")

labels_df = pd.read_csv(LABELS_CSV)

print("Labels shape:", labels_df.shape)
display(labels_df.head())

print("\nClass distribution:")
print(labels_df["label"].value_counts())

print("\nClass proportions:")
print(labels_df["label"].value_counts(normalize=True))

In [ ]:
# Create a stratified subset of 5000 images to ensure balanced class representation

SUBSET_SIZE = 5000

subset_df, _ = train_test_split(
    labels_df,
    train_size=SUBSET_SIZE,
    stratify=labels_df["label"],
    random_state=42
)

subset_df = subset_df.reset_index(drop=True)

print("Subset size:", subset_df.shape)
print("\nSubset class distribution:")
print(subset_df["label"].value_counts())
print("\nSubset class proportions:")
print(subset_df["label"].value_counts(normalize=True))

In [ ]:
# Split the subset into training, validation, and test sets using stratified sampling

train_df, temp_df = train_test_split(
    subset_df,
    test_size=0.30,
    stratify=subset_df["label"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train size:", len(train_df))
print("Validation size:", len(val_df))
print("Test size:", len(test_df))

## Image Preprocessing and Augmentation

Image preprocessing and augmentation are essential steps to improve model generalization and performance.

- **Normalization** ensures that pixel values are scaled appropriately for stable training.
- **Data augmentation** artificially increases dataset diversity by applying random transformations.

In this project, we define:
- A **baseline transform** (no augmentation)
- **Augmentation Strategy 1**: Horizontal flipping and rotation
- **Augmentation Strategy 2**: Horizontal flipping and color jitter

These augmentation strategies will be used in different training experiments to analyze their impact on model performance.

In [ ]:
# Define image transformations including normalization and two augmentation strategies

from torchvision import transforms

base_transform = transforms.Compose([
    transforms.Resize((96, 96)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                         std=[0.5, 0.5, 0.5])
])

augment_transform_1 = transforms.Compose([
    transforms.Resize((96, 96)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                         std=[0.5, 0.5, 0.5])
])

augment_transform_2 = transforms.Compose([
    transforms.Resize((96, 96)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                         std=[0.5, 0.5, 0.5])
])

## Custom Dataset and DataLoaders

To train deep learning models in PyTorch, the image data must be wrapped inside a custom `Dataset` class. This class is responsible for:
- Loading image files from disk
- Mapping each image to its corresponding label
- Applying the selected preprocessing or augmentation transform

After defining the dataset class, `DataLoader` objects are created for the training, validation, and test sets. These dataloaders handle batching, shuffling, and efficient iteration during training and evaluation.

In [ ]:
# Define a custom PyTorch Dataset class to load histopathology images and their binary labels

class HistopathDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_id = self.df.iloc[idx]["id"]
        label = self.df.iloc[idx]["label"]

        img_path = os.path.join(self.img_dir, img_id + ".tif")
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.float32)

In [ ]:
# Create dataset objects for training, validation, and testing using the first augmentation strategy for training

train_dataset = HistopathDataset(train_df, TRAIN_DIR, transform=augment_transform_1)
val_dataset = HistopathDataset(val_df, TRAIN_DIR, transform=base_transform)
test_dataset = HistopathDataset(test_df, TRAIN_DIR, transform=base_transform)

print("Train dataset size:", len(train_dataset))
print("Validation dataset size:", len(val_dataset))
print("Test dataset size:", len(test_dataset))

In [ ]:
# Create dataloaders for batching and shuffling data during training and evaluation

BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print("DataLoaders created successfully")

## Model 1: Custom CNN

The first model used in this project is a custom Convolutional Neural Network (CNN) built from scratch. This model satisfies the project requirement that at least one model must be designed by us.

The architecture consists of:
- Multiple convolutional layers to extract spatial features
- ReLU activation for non-linearity
- Max pooling layers for downsampling
- Fully connected layers for binary classification

This custom CNN serves as the baseline deep learning model for cancer image classification.

In [ ]:
# Define the computation device so training can use GPU if available, otherwise CPU

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
# Define an improved custom CNN with BatchNorm for stronger feature learning

class CustomCNN(nn.Module):
    def __init__(self):
        super(CustomCNN, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 96 -> 48

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 48 -> 24

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),   # 24 -> 12

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2)    # 12 -> 6
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 6 * 6, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 1)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

## Training Configuration for Custom CNN

To train the custom CNN, a binary classification setup is used. Since the task is to classify each image patch as either cancerous or non-cancerous, the model outputs a single logit value.

The following training components are used in the first experiment:

- **Loss Function:** Binary Cross-Entropy with Logits (`BCEWithLogitsLoss`)
- **Optimizer:** Adam
- **Learning Rate Scheduler:** StepLR

In addition to loss, training and validation accuracy are also tracked at each epoch to better understand model learning behavior.

In [ ]:
# Define improved optimizer and scheduler settings for the stronger custom CNN

model = CustomCNN().to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)

print("Improved Custom CNN setup is ready")

In [ ]:
# Define the training and validation loop to track both loss and accuracy across epochs

def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, epochs=6):
    train_losses = []
    val_losses = []
    train_accuracies = []
    val_accuracies = []

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        train_correct = 0
        train_total = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.unsqueeze(1).to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

            probs = torch.sigmoid(outputs)
            preds = (probs >= 0.5).float()

            train_correct += (preds == labels).sum().item()
            train_total += labels.size(0)

        scheduler.step()

        train_loss = running_loss / len(train_loader)
        train_acc = train_correct / train_total

        train_losses.append(train_loss)
        train_accuracies.append(train_acc)

        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                labels = labels.unsqueeze(1).to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item()

                probs = torch.sigmoid(outputs)
                preds = (probs >= 0.5).float()

                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)

        val_loss = val_loss / len(val_loader)
        val_acc = val_correct / val_total

        val_losses.append(val_loss)
        val_accuracies.append(val_acc)

        print(
            f"Epoch [{epoch+1}/{epochs}] - "
            f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, "
            f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}"
        )

    return train_losses, val_losses, train_accuracies, val_accuracies

## Custom CNN Experiment 1: Adam + StepLR + Augmentation Strategy 1

The first training experiment for the custom CNN uses:

- **Augmentation Strategy 1:** Horizontal flip and rotation
- **Optimizer:** Adam
- **Scheduler:** StepLR

This experiment serves as the baseline configuration for the custom CNN. The model is trained for multiple epochs, and both loss and accuracy are tracked on the training and validation sets.

In [ ]:
# Train the first Custom CNN experiment and store loss and accuracy values

train_losses, val_losses, train_accuracies, val_accuracies = train_model(
    model,
    train_loader,
    val_loader,
    criterion,
    optimizer,
    scheduler,
    epochs=15
)

## Hyperparameter Tuning for the Custom CNN

To improve the performance of the custom CNN, hyperparameter tuning is performed. Instead of changing too many factors at once, a controlled search is used over a few important training parameters.

The main hyperparameters tuned in this section are:
- Learning rate
- Weight decay

The best configuration is selected based on validation ROC-AUC and validation accuracy.

In [ ]:
# Define a helper function to evaluate a model on the validation set using accuracy and ROC-AUC

def evaluate_model(model, val_loader):
    model.eval()

    all_labels = []
    all_probs = []
    all_preds = []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            probs = torch.sigmoid(outputs).squeeze(1)
            preds = (probs >= 0.5).int()

            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    val_acc = accuracy_score(all_labels, all_preds)
    val_auc = roc_auc_score(all_labels, all_probs)

    return val_acc, val_auc

In [ ]:
# Perform hyperparameter tuning over different learning rates and weight decay values

learning_rates = [1e-3, 5e-4, 1e-4]
weight_decays = [0.0, 1e-4, 1e-3]

tuning_results = []

for lr in learning_rates:
    for wd in weight_decays:
        print(f"\nTraining with lr={lr}, weight_decay={wd}")

        model_tune = CustomCNN().to(device)

        criterion_tune = nn.BCEWithLogitsLoss()
        optimizer_tune = optim.Adam(model_tune.parameters(), lr=lr, weight_decay=wd)
        scheduler_tune = optim.lr_scheduler.StepLR(optimizer_tune, step_size=3, gamma=0.5)

        train_losses_t, val_losses_t, train_acc_t, val_acc_t = train_model(
            model_tune,
            train_loader,
            val_loader,
            criterion_tune,
            optimizer_tune,
            scheduler_tune,
            epochs=5
        )

        final_val_acc, final_val_auc = evaluate_model(model_tune, val_loader)

        tuning_results.append({
            "learning_rate": lr,
            "weight_decay": wd,
            "final_val_accuracy": final_val_acc,
            "final_val_auc": final_val_auc
        })

        print(f"Validation Accuracy: {final_val_acc:.4f}")
        print(f"Validation ROC-AUC: {final_val_auc:.4f}")

In [ ]:
# Convert tuning results into a dataframe and sort by validation ROC-AUC

tuning_df = pd.DataFrame(tuning_results)
tuning_df = tuning_df.sort_values(by="final_val_auc", ascending=False).reset_index(drop=True)

tuning_df

In [ ]:
# Display the best hyperparameter combination found during tuning

best_config = tuning_df.iloc[0]

print("Best Hyperparameter Configuration:")
print(best_config)

## Final Custom CNN Configuration

Based on hyperparameter tuning, the best Custom CNN configuration used:

- Learning rate = 0.001
- Weight decay = 0.0001

This tuned configuration is used as the final version of the custom-built model for further evaluation and comparison.

In [ ]:
# Train the final tuned Custom CNN using the best hyperparameter configuration

final_custom_model = CustomCNN().to(device)

final_criterion = nn.BCEWithLogitsLoss()
final_optimizer = optim.Adam(final_custom_model.parameters(), lr=0.001, weight_decay=0.0001)
final_scheduler = optim.lr_scheduler.StepLR(final_optimizer, step_size=3, gamma=0.5)

final_train_losses, final_val_losses, final_train_accuracies, final_val_accuracies = train_model(
    final_custom_model,
    train_loader,
    val_loader,
    final_criterion,
    final_optimizer,
    final_scheduler,
    epochs=10
)

In [ ]:
# Evaluate the tuned Custom CNN on the validation set and compute final validation metrics

final_custom_model.eval()

final_val_labels = []
final_val_probs = []
final_val_preds = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = final_custom_model(images)
        probs = torch.sigmoid(outputs).squeeze(1)
        preds = (probs >= 0.5).int()

        final_val_labels.extend(labels.cpu().numpy())
        final_val_probs.extend(probs.cpu().numpy())
        final_val_preds.extend(preds.cpu().numpy())

final_val_acc = accuracy_score(final_val_labels, final_val_preds)
final_val_auc = roc_auc_score(final_val_labels, final_val_probs)

print("Final Custom CNN Validation Accuracy:", round(final_val_acc, 4))
print("Final Custom CNN Validation ROC-AUC:", round(final_val_auc, 4))

In [ ]:
# Plot the ROC curve for the tuned Custom CNN on the validation set

final_fpr, final_tpr, final_thresholds = roc_curve(final_val_labels, final_val_probs)

plt.figure(figsize=(7, 5))
plt.plot(final_fpr, final_tpr, label=f"Final Custom CNN (AUC = {final_val_auc:.4f})")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Final Tuned Custom CNN")
plt.legend()
plt.show()

## Interpretation of the Final Custom CNN

The tuned Custom CNN achieved its best validation performance around Epoch 9, reaching a validation accuracy of approximately 86.27%. This suggests that the selected hyperparameters improved model learning compared to the earlier baseline configuration.

A slight decline in validation performance after the best epoch indicates mild overfitting, which is common in deep learning models trained on smaller subsets. Overall, the tuned Custom CNN provides a strong built-from-scratch baseline for comparison with the transfer learning model in the next section.

## Additional Tuning and Comparison with a Second Optimizer

To satisfy the project requirement of testing at least two optimizers and two schedulers, the custom CNN is also trained using a second configuration based on SGD.

This second experiment uses:
- **Optimizer:** SGD
- **Scheduler:** CosineAnnealingLR
- **Augmentation Strategy 2:** Horizontal flip and color jitter

The goal is to compare this configuration against the Adam-based setup and analyze how optimizer choice affects model performance.

In [ ]:
# Create training data for the second augmentation strategy used with the SGD-based experiment

train_dataset_sgd = HistopathDataset(train_df, TRAIN_DIR, transform=augment_transform_2)
train_loader_sgd = DataLoader(train_dataset_sgd, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

print("SGD training DataLoader is ready")

In [ ]:
# Define a fresh Custom CNN model with SGD optimizer and CosineAnnealingLR scheduler

sgd_model = CustomCNN().to(device)

criterion_sgd = nn.BCEWithLogitsLoss()
optimizer_sgd = optim.SGD(sgd_model.parameters(), lr=0.001, momentum=0.9, weight_decay=1e-4)
scheduler_sgd = optim.lr_scheduler.CosineAnnealingLR(optimizer_sgd, T_max=10)

print("SGD + CosineAnnealingLR setup is ready")

In [ ]:
# Train the Custom CNN using the SGD-based configuration and track loss and accuracy

sgd_train_losses, sgd_val_losses, sgd_train_accuracies, sgd_val_accuracies = train_model(
    sgd_model,
    train_loader_sgd,
    val_loader,
    criterion_sgd,
    optimizer_sgd,
    scheduler_sgd,
    epochs=10
)

## Hyperparameter Tuning for the SGD-Based Custom CNN

In addition to the Adam-based tuning, a second tuning process is performed for the SGD-based Custom CNN. This ensures that both optimizers are evaluated fairly.

For the SGD configuration, the following hyperparameters are tuned:
- Learning rate
- Weight decay

The model is trained under multiple SGD settings, and the best configuration is selected based on validation ROC-AUC and validation accuracy.

In [ ]:
# Perform hyperparameter tuning for the SGD-based Custom CNN using different learning rates and weight decay values

sgd_learning_rates = [0.01, 0.001]
sgd_weight_decays = [0.0, 1e-4]

sgd_tuning_results = []

for lr in sgd_learning_rates:
    for wd in sgd_weight_decays:
        print(f"\nTraining SGD model with lr={lr}, weight_decay={wd}")

        model_sgd_tune = CustomCNN().to(device)

        criterion_sgd_tune = nn.BCEWithLogitsLoss()
        optimizer_sgd_tune = optim.SGD(
            model_sgd_tune.parameters(),
            lr=lr,
            momentum=0.9,
            weight_decay=wd
        )
        scheduler_sgd_tune = optim.lr_scheduler.CosineAnnealingLR(
            optimizer_sgd_tune,
            T_max=5
        )

        train_losses_tmp, val_losses_tmp, train_acc_tmp, val_acc_tmp = train_model(
            model_sgd_tune,
            train_loader_sgd,
            val_loader,
            criterion_sgd_tune,
            optimizer_sgd_tune,
            scheduler_sgd_tune,
            epochs=5
        )

        val_acc_tmp_final, val_auc_tmp_final = evaluate_model(model_sgd_tune, val_loader)

        sgd_tuning_results.append({
            "learning_rate": lr,
            "weight_decay": wd,
            "final_val_accuracy": val_acc_tmp_final,
            "final_val_auc": val_auc_tmp_final
        })

        print(f"Validation Accuracy: {val_acc_tmp_final:.4f}")
        print(f"Validation ROC-AUC: {val_auc_tmp_final:.4f}")

In [ ]:
# Convert the SGD tuning results into a dataframe and sort by validation ROC-AUC

sgd_tuning_df = pd.DataFrame(sgd_tuning_results)
sgd_tuning_df = sgd_tuning_df.sort_values(by="final_val_auc", ascending=False).reset_index(drop=True)

sgd_tuning_df

In [ ]:
# Display the best hyperparameter configuration found for the SGD-based Custom CNN

best_sgd_config = sgd_tuning_df.iloc[0]

print("Best SGD Hyperparameter Configuration:")
print(best_sgd_config)

## Final SGD-Based Custom CNN Configuration

Based on the SGD tuning results, the best SGD configuration used:

- Learning rate = 0.01
- Weight decay = 0.0
- Optimizer = SGD
- Scheduler = CosineAnnealingLR
- Augmentation Strategy 2 = Horizontal flip and color jitter

This final SGD-based Custom CNN is trained and evaluated for comparison against the tuned Adam-based Custom CNN.

In [ ]:
# Train the final SGD-based Custom CNN using the best SGD hyperparameter configuration

final_sgd_model = CustomCNN().to(device)

final_sgd_criterion = nn.BCEWithLogitsLoss()
final_sgd_optimizer = optim.SGD(
    final_sgd_model.parameters(),
    lr=0.01,
    momentum=0.9,
    weight_decay=0.0
)
final_sgd_scheduler = optim.lr_scheduler.CosineAnnealingLR(final_sgd_optimizer, T_max=10)

final_sgd_train_losses, final_sgd_val_losses, final_sgd_train_accuracies, final_sgd_val_accuracies = train_model(
    final_sgd_model,
    train_loader_sgd,
    val_loader,
    final_sgd_criterion,
    final_sgd_optimizer,
    final_sgd_scheduler,
    epochs=10
)

In [ ]:
# Evaluate the final SGD-based Custom CNN on the validation set and compute final metrics

final_sgd_model.eval()

final_sgd_val_labels = []
final_sgd_val_probs = []
final_sgd_val_preds = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = final_sgd_model(images)
        probs = torch.sigmoid(outputs).squeeze(1)
        preds = (probs >= 0.5).int()

        final_sgd_val_labels.extend(labels.cpu().numpy())
        final_sgd_val_probs.extend(probs.cpu().numpy())
        final_sgd_val_preds.extend(preds.cpu().numpy())

final_sgd_val_acc = accuracy_score(final_sgd_val_labels, final_sgd_val_preds)
final_sgd_val_auc = roc_auc_score(final_sgd_val_labels, final_sgd_val_probs)

print("Final SGD Custom CNN Validation Accuracy:", round(final_sgd_val_acc, 4))
print("Final SGD Custom CNN Validation ROC-AUC:", round(final_sgd_val_auc, 4))

In [ ]:
# Plot the ROC curve for the final SGD-based Custom CNN

final_sgd_fpr, final_sgd_tpr, final_sgd_thresholds = roc_curve(final_sgd_val_labels, final_sgd_val_probs)

plt.figure(figsize=(7, 5))
plt.plot(final_sgd_fpr, final_sgd_tpr, label=f"Final SGD Custom CNN (AUC = {final_sgd_val_auc:.4f})")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Final SGD Custom CNN")
plt.legend()
plt.show()

In [ ]:
# Compare the final Adam-based and final SGD-based Custom CNN models in a summary table

custom_cnn_results_df = pd.DataFrame({
    "Model Configuration": [
        "Custom CNN + Adam + StepLR + Augmentation 1",
        "Custom CNN + SGD + CosineAnnealingLR + Augmentation 2"
    ],
    "Validation Accuracy": [final_val_acc, final_sgd_val_acc],
    "Validation ROC-AUC": [final_val_auc, final_sgd_val_auc]
})

custom_cnn_results_df

In [ ]:
# Plot ROC curves of the final Adam-based and SGD-based Custom CNN models on the same figure

plt.figure(figsize=(8, 6))
plt.plot(final_fpr, final_tpr, label=f"Adam-based Custom CNN (AUC = {final_val_auc:.4f})")
plt.plot(final_sgd_fpr, final_sgd_tpr, label=f"SGD-based Custom CNN (AUC = {final_sgd_val_auc:.4f})")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Comparison - Final Custom CNN Configurations")
plt.legend()
plt.show()

## ROC Curve Comparison – Custom CNN Models

The ROC curve comparison illustrates the performance of the two optimized Custom CNN configurations trained using different optimizers.

### Key Observations:

- The **Adam-based Custom CNN** achieves a higher ROC-AUC score (**~0.935**), indicating better overall discrimination between cancerous and non-cancerous samples.
- The **SGD-based Custom CNN** achieves a slightly lower ROC-AUC (**~0.907**), suggesting comparatively weaker performance.
- The ROC curve for the Adam model consistently lies above the SGD curve, confirming its superior predictive capability across different classification thresholds.
- Both models perform significantly better than the random baseline (diagonal dashed line), indicating that the models are learning meaningful patterns from the data.

### Conclusion:

The **Adam optimizer combined with StepLR scheduling and augmentation strategy 1** provides the best performance for the custom-built CNN. This configuration is selected as the final Custom CNN model for comparison with the transfer learning model in the next section.

## Transfer Learning Model: DenseNet121

In addition to the custom CNN, a pretrained DenseNet121 model is used for transfer learning.

DenseNet (Densely Connected Convolutional Networks) is particularly well-suited for medical imaging tasks due to:

- Efficient feature reuse through dense connections
- Improved gradient flow compared to traditional CNNs
- Strong performance in image classification tasks involving fine-grained patterns

DenseNet121 is selected because it provides a good balance between model complexity and performance.

The model is initialized with pretrained ImageNet weights and modified for binary classification of cancer detection.

## Training Strategy for DenseNet121

To ensure a fair comparison with the custom CNN, the DenseNet121 transfer learning model is also evaluated under multiple training configurations.

The DenseNet experiments include:
- Two optimizers: Adam and SGD
- Two learning rate schedulers: StepLR and CosineAnnealingLR
- Two augmentation strategies
- Hyperparameter tuning over learning rate and weight decay

This allows comparison not only between model architectures, but also between training strategies across both models.

## Model 2: DenseNet121

To complement the custom-built CNN, a transfer learning model is used as the second architecture in this project.

DenseNet121 is selected because:
- it encourages feature reuse through dense connections
- it improves gradient flow during training
- it has shown strong performance in image classification tasks, including medical imaging

For this binary classification problem, the pretrained DenseNet121 model is initialized with ImageNet weights, and its final classifier layer is replaced with a single output neuron for cancer detection.

In [ ]:
# Create training and validation datasets for the first DenseNet experiment using augmentation strategy 1

train_dataset_dn_1 = HistopathDataset(train_df, TRAIN_DIR, transform=augment_transform_1)
val_dataset_dn = HistopathDataset(val_df, TRAIN_DIR, transform=base_transform)
test_dataset_dn = HistopathDataset(test_df, TRAIN_DIR, transform=base_transform)

train_loader_dn_1 = DataLoader(train_dataset_dn_1, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader_dn = DataLoader(val_dataset_dn, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader_dn = DataLoader(test_dataset_dn, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print("DenseNet DataLoaders for Experiment 1 are ready")

In [ ]:
# Load pretrained DenseNet121 and replace the classifier for binary classification

densenet_model = models.densenet121(pretrained=True)

num_features = densenet_model.classifier.in_features
densenet_model.classifier = nn.Linear(num_features, 1)

densenet_model = densenet_model.to(device)

print(densenet_model)

In [ ]:
# Define loss function, optimizer, and scheduler for DenseNet Experiment 1 using Adam and StepLR

criterion_dn = nn.BCEWithLogitsLoss()
optimizer_dn = optim.Adam(densenet_model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler_dn = optim.lr_scheduler.StepLR(optimizer_dn, step_size=3, gamma=0.5)

print("DenseNet Experiment 1 setup is ready")

## DenseNet121 Experiment 1: Adam + StepLR + Augmentation Strategy 1

The first DenseNet121 experiment uses the following configuration:

- **Augmentation Strategy 1:** Random horizontal flip and rotation
- **Optimizer:** Adam
- **Scheduler:** StepLR

This experiment serves as the baseline transfer learning configuration and will be compared against the custom CNN as well as additional DenseNet training setups.

In [ ]:
# Train DenseNet121 using Adam, StepLR, and augmentation strategy 1

dn1_train_losses, dn1_val_losses, dn1_train_accuracies, dn1_val_accuracies = train_model(
    densenet_model,
    train_loader_dn_1,
    val_loader_dn,
    criterion_dn,
    optimizer_dn,
    scheduler_dn,
    epochs=6
)

## Hyperparameter Tuning for DenseNet121 (Adam Configuration)

To improve the performance of the DenseNet121 model, hyperparameter tuning is performed.

The following parameters are tuned:
- Learning rate
- Weight decay

The best configuration is selected based on validation ROC-AUC and accuracy.

In [ ]:
# Hyperparameter tuning for DenseNet (Adam + StepLR)

dn_learning_rates = [1e-4, 5e-5]
dn_weight_decays = [0.0, 1e-4]

dn_tuning_results = []

for lr in dn_learning_rates:
    for wd in dn_weight_decays:
        print(f"\nTraining DenseNet with lr={lr}, weight_decay={wd}")

        model_dn_tune = models.densenet121(pretrained=True)
        num_features = model_dn_tune.classifier.in_features
        model_dn_tune.classifier = nn.Linear(num_features, 1)
        model_dn_tune = model_dn_tune.to(device)

        optimizer_dn_tune = optim.Adam(model_dn_tune.parameters(), lr=lr, weight_decay=wd)
        scheduler_dn_tune = optim.lr_scheduler.StepLR(optimizer_dn_tune, step_size=3, gamma=0.5)

        train_model(
            model_dn_tune,
            train_loader_dn_1,
            val_loader_dn,
            nn.BCEWithLogitsLoss(),
            optimizer_dn_tune,
            scheduler_dn_tune,
            epochs=3   # keep small for tuning
        )

        val_acc_tmp, val_auc_tmp = evaluate_model(model_dn_tune, val_loader_dn)

        dn_tuning_results.append({
            "lr": lr,
            "weight_decay": wd,
            "val_acc": val_acc_tmp,
            "val_auc": val_auc_tmp
        })

        print(f"Val Acc: {val_acc_tmp:.4f}, Val AUC: {val_auc_tmp:.4f}")

In [ ]:
# Show tuning results

dn_tuning_df = pd.DataFrame(dn_tuning_results)
dn_tuning_df = dn_tuning_df.sort_values(by="val_auc", ascending=False).reset_index(drop=True)

dn_tuning_df

In [ ]:
# Best DenseNet configuration

best_dn_config = dn_tuning_df.iloc[0]
print("Best DenseNet Config:")
print(best_dn_config)

## Final DenseNet121 Configuration (Adam-Based)

Based on hyperparameter tuning, the best Adam-based DenseNet121 configuration used:

- Learning rate = 0.0001
- Weight decay = 0.0
- Optimizer = Adam
- Scheduler = StepLR
- Augmentation Strategy 1 = Random horizontal flip and rotation

This tuned configuration is used as the final Adam-based DenseNet121 model for evaluation and later comparison with the SGD-based DenseNet configuration.

In [ ]:
# Train the final Adam-based DenseNet121 using the best hyperparameter configuration

final_dn_model = models.densenet121(pretrained=True)
num_features = final_dn_model.classifier.in_features
final_dn_model.classifier = nn.Linear(num_features, 1)
final_dn_model = final_dn_model.to(device)

final_dn_criterion = nn.BCEWithLogitsLoss()
final_dn_optimizer = optim.Adam(final_dn_model.parameters(), lr=1e-4, weight_decay=0.0)
final_dn_scheduler = optim.lr_scheduler.StepLR(final_dn_optimizer, step_size=3, gamma=0.5)

final_dn_train_losses, final_dn_val_losses, final_dn_train_accuracies, final_dn_val_accuracies = train_model(
    final_dn_model,
    train_loader_dn_1,
    val_loader_dn,
    final_dn_criterion,
    final_dn_optimizer,
    final_dn_scheduler,
    epochs=6
)

In [ ]:
# Plot the training and validation loss and accuracy curves for the final Adam-based DenseNet121 model

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(final_dn_train_losses, label="Train Loss")
plt.plot(final_dn_val_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Final DenseNet121 (Adam): Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(final_dn_train_accuracies, label="Train Accuracy")
plt.plot(final_dn_val_accuracies, label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Final DenseNet121 (Adam): Accuracy")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Evaluate the final Adam-based DenseNet121 on the validation set and compute validation accuracy and ROC-AUC

final_dn_model.eval()

final_dn_val_labels = []
final_dn_val_probs = []
final_dn_val_preds = []

with torch.no_grad():
    for images, labels in val_loader_dn:
        images = images.to(device)
        labels = labels.to(device)

        outputs = final_dn_model(images)
        probs = torch.sigmoid(outputs).squeeze(1)
        preds = (probs >= 0.5).int()

        final_dn_val_labels.extend(labels.cpu().numpy())
        final_dn_val_probs.extend(probs.cpu().numpy())
        final_dn_val_preds.extend(preds.cpu().numpy())

final_dn_val_acc = accuracy_score(final_dn_val_labels, final_dn_val_preds)
final_dn_val_auc = roc_auc_score(final_dn_val_labels, final_dn_val_probs)

print("Final DenseNet121 (Adam) Validation Accuracy:", round(final_dn_val_acc, 4))
print("Final DenseNet121 (Adam) Validation ROC-AUC:", round(final_dn_val_auc, 4))

In [ ]:
# Plot the ROC curve for the final Adam-based DenseNet121 model

final_dn_fpr, final_dn_tpr, final_dn_thresholds = roc_curve(final_dn_val_labels, final_dn_val_probs)

plt.figure(figsize=(7, 5))
plt.plot(final_dn_fpr, final_dn_tpr, label=f"Final DenseNet121 (Adam) AUC = {final_dn_val_auc:.4f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Final DenseNet121 (Adam)")
plt.legend()
plt.show()

## Final Model Comparison: Custom CNN vs DenseNet121

This section compares the performance of the best Custom CNN configuration and the best DenseNet121 model.

The comparison is based on:
- Validation Accuracy
- Validation ROC-AUC

This helps evaluate the impact of transfer learning versus training a model from scratch.

In [ ]:
# Create a comparison table for Custom CNN vs DenseNet121

comparison_df = pd.DataFrame({
    "Model": ["Custom CNN (Best Adam)", "DenseNet121 (Adam)"],
    "Validation Accuracy": [final_val_acc, final_dn_val_acc],
    "Validation ROC-AUC": [final_val_auc, final_dn_val_auc]
})

comparison_df

In [ ]:
# Plot comparison of Validation Accuracy and ROC-AUC

import numpy as np

models_list = comparison_df["Model"]
accuracy_vals = comparison_df["Validation Accuracy"]
auc_vals = comparison_df["Validation ROC-AUC"]

x = np.arange(len(models_list))

plt.figure(figsize=(8, 5))

plt.bar(x - 0.2, accuracy_vals, width=0.4, label="Accuracy")
plt.bar(x + 0.2, auc_vals, width=0.4, label="ROC-AUC")

plt.xticks(x, models_list, rotation=10)
plt.ylabel("Score")
plt.title("Model Comparison: Custom CNN vs DenseNet121")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Plot ROC curves for Custom CNN and DenseNet121 together

plt.figure(figsize=(8, 6))

plt.plot(final_fpr, final_tpr, label=f"Custom CNN (AUC = {final_val_auc:.4f})")
plt.plot(final_dn_fpr, final_dn_tpr, label=f"DenseNet121 (AUC = {final_dn_val_auc:.4f})")

plt.plot([0, 1], [0, 1], linestyle="--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison: Custom CNN vs DenseNet121")
plt.legend()

plt.show()

## Final Comparison and Conclusion

The comparison between the Custom CNN and DenseNet121 highlights the effectiveness of transfer learning.

### Key Findings:
- DenseNet121 achieved higher validation accuracy and ROC-AUC compared to the Custom CNN.
- The Custom CNN performed reasonably well but was limited by training from scratch on a smaller dataset.
- DenseNet121 benefited from pretrained ImageNet features, enabling better generalization.

### Final Conclusion:
Transfer learning with DenseNet121 significantly improves performance for histopathologic cancer detection tasks compared to a custom-built CNN.